In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(delivery_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
experience_column = df['Courier_Experience_yrs']
df['Courier_Experience_yrs'] = experience_column.fillna(experience_column.sum() / len(experience_column))
df = df.dropna()

In [ ]:
# Task 3: Write your code here:
df = df[df.duplicated() == False]

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
le = LabelEncoder()

df['Weather'] = le.fit_transform(df['Weather'])
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df['Time_of_Day'] = le.fit_transform(df['Time_of_Day'])
df['Vehicle_Type'] = le.fit_transform(df['Vehicle_Type'])
df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
std = StandardScaler()
ndf = std.fit_transform(df.drop(columns=['Delivery_Time']))

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import StratifiedKFold
X = pd.DataFrame(std.fit_transform(df.drop(columns=['Delivery_Time'])))
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestRegressor(n_estimators=200)
errors = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    errors.append((1/len(y)) * np.sum(np.abs(y_test - y_pred)))

print("Average error (MAE):", sum(errors)/len(errors))

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(rf.feature_importances_, edgecolor='black')
plt.show()
print(rf.feature_importances_)

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:
%pip install catboost
from catboost import CatBoostRegressor
rfr = RandomForestRegressor(n_estimators=200)
cbr = CatBoostRegressor(verbose=0)
rfr_errors = []
cbr_errors = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rfr.fit(X_train, y_train)
    cbr.fit(X_train, y_train)

    rfr_pred = rfr.predict(X_test)
    cbr_pred = cbr.predict(X_test)

    rfr_errors.append((1/len(y)) * np.sum(np.abs(y_test - rfr_pred)))
    cbr_errors.append((1/len(y)) * np.sum(np.abs(y_test - cbr_pred)))

print("Average RandomForestRegressor error (MAE):", sum(rfr_errors)/len(rfr_errors))
print("Average CatBoostRegressor error (MAE):", sum(cbr_errors)/len(cbr_errors))